# GPW: two capacities and three stitch modes

FLIGHT800k,TCO.L2 and ICETHICK,rank64 only. Two explicit configurations:
8 patches x32 points (256 total),8 stitches per connected pair;
4 patches x256 points (1024 total),16 stitches per connected pair.
Each compares none/fixed/learned stitches. None activates zero constraints.
These are NOT size-matched configurations: capacity,patch count and stitch
density all change. Compare stitch modes within each configuration first.

SEEDS=[31] gives18 runs. Change/add seeds directly,for example[31,32,33]
gives54 runs. Data/subset/split seed remains31. FLIGHT samples800,000 rows
without replacement,80/20 split;TCO.L2/ICETHICK use full approved datasets.
Attach airline.npz,airline.json,icethk.txt;enable Internet and a GPU.

Both configurations reuse exactly the same verified prepared arrays/splits.
Within each capacity/dataset/seed,the three modes share initial supports,
centers,candidate graph and candidate stitches. Initialization is NOT held
identical across capacities: there are different numbers of points/patches.
Local kernels,q family,rank64 and existing prior-dependent q initialization
remain;100 epochs/20 warm-up and existing stopping/numerics/optimizers remain.

Each configuration is an explicit independent fresh-worker batch. Completed
models save a ZIP before the next model;each completed configuration also
saves its collection ZIP before the next configuration. Failures stop the
schedule;there is no auto-skip,resume,device change or time truncation.
The earlier6-9-hour estimate DOES NOT apply to these18 runs. This schedule
has no measured duration or14-hour guarantee. Review the finite plan first.
Save/download outputs externally;session files are not an external backup.


## 1. Load latest source and dependencies


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['checkout', 'install', 'settings', 'preflight', 'data', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in []:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
import hashlib, importlib, json, os, subprocess, sys
from pathlib import Path

REPOSITORY = 'https://github.com/chahineNejm/Generalized-Patchwork.git'
BRANCH = 'main'
ROOT = Path('/kaggle/working/gpw_source')
os.environ["GIT_TERMINAL_PROMPT"] = "0"
if ROOT.exists():
    top = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "--show-toplevel"], text=True).strip()
    if Path(top).resolve() != ROOT.resolve():
        raise ValueError("Existing checkout must be its own Git repository")
    remote = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
    branch = subprocess.check_output(["git", "-C", str(ROOT), "branch", "--show-current"], text=True).strip()
    if remote != REPOSITORY or branch != BRANCH:
        raise ValueError("Existing checkout repository or branch does not match the requested source")
    subprocess.run(["git", "-C", str(ROOT), "diff", "--exit-code", "HEAD"], check=True)
    untracked = subprocess.check_output(["git", "-C", str(ROOT), "ls-files", "--others",
        "--exclude-standard", "--", ":(glob)src/**/*.py", ":(glob)experiments/**/*.py"], text=True).strip()
    if untracked:
        raise ValueError(f"Untracked source files prevent verified reuse: {untracked}")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(ROOT), "merge-base", "--is-ancestor", "HEAD", "origin/" + BRANCH], check=True)
    subprocess.run(["git", "-C", str(ROOT), "merge", "--ff-only", "origin/" + BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH,
                    "--single-branch", REPOSITORY, str(ROOT)], check=True)
subprocess.run(["git", "-C", str(ROOT), "sparse-checkout", "set", "--no-cone",
                "/src/", "/experiments/", "/pyproject.toml", "/README.md"], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", BRANCH], check=True)

COMMIT = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
SOURCE_ZIP = subprocess.check_output(["git", "-C", str(ROOT), "archive", "--format=zip",
                                      COMMIT, "pyproject.toml", "README.md", "src", "experiments"])
SOURCE_PROVENANCE = {"repository": REPOSITORY, "branch": BRANCH, "commit": COMMIT,
                     "source_sha256": hashlib.sha256(SOURCE_ZIP).hexdigest()}
os.environ["MPLCONFIGDIR"] = str(ROOT / "results/.matplotlib")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.chdir(ROOT)
for path in reversed((str(ROOT / "src"), str(ROOT))):
    if path not in sys.path:
        sys.path.insert(0, path)
importlib.invalidate_caches()
_GPW_CELL_RESULT = print(SOURCE_PROVENANCE)
_OISST_STAGES['checkout'] = True
_GPW_CELL_RESULT


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['install', 'settings', 'preflight', 'data', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['checkout']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "--editable", str(ROOT) + "[data,ukhousing,benchmark]"], check=True)
import runpy
run_worker = runpy.run_path(str(ROOT / "experiments/colab_client.py"))["run_worker"]
WORK_ROOT = Path("/kaggle/working/gpw_ablation_results")
DATA_ROOT = Path("/kaggle/working/gpw_data")

def call_worker(action, *, on_event=None, **values):
    return run_worker(ROOT, {"action": action, "source_provenance": SOURCE_PROVENANCE, **values},
        module="experiments.multidataset_worker", jobs_directory=WORK_ROOT / "worker_jobs", on_event=on_event)
_OISST_STAGES['install'] = True


## 2. Configure capacities,seeds and datasets

Edit SEEDS directly. CONFIGURATIONS can explicitly select either/both sizes;
stitches_per_edge applies to fixed/learned only. MODEL_OVERRIDES applies
equally to all modes/sizes per dataset;it cannot override stitches_per_edge.
Review the printed total before RUN_TRAINING=True. No benchmark is launched
by this settings stage. Data are prepared once and verified for both plans.


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['settings', 'preflight', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['install']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
from datetime import UTC, datetime
from IPython.display import FileLink, Image, display
import json
import re
import pandas as pd

SEEDS = [31]
CONFIGURATIONS = [{'name': '8x32_stitches8', 'inducing_points': 256, 'patches': 8, 'stitches_per_edge': 8}, {'name': '4x256_stitches16', 'inducing_points': 1024, 'patches': 4, 'stitches_per_edge': 16}]
DATASETS = ['flight', 'tco_l2', 'icethick']
INPUT_ROOT = "/kaggle/input/datasets/chahinenejma/gpw-kaggle-inputs"
RANKS = [64]
STITCH_MODES = ['none', 'fixed', 'learned']
EPOCHS = 100
WARMUP_EPOCHS = 20
EARLY_STOPPING = {'patience': 10, 'relative_min_improvement': 0.001}
DATA_OVERRIDES = {'flight': {'subset_size': 800000}}
MODEL_OVERRIDES = {}
SAVE_INDUCING_POSITIONS = True
PROGRESS_EVERY_EPOCHS = 20

if not isinstance(CONFIGURATIONS, list) or not CONFIGURATIONS:
    raise ValueError("CONFIGURATIONS must explicitly select at least one capacity")
if set(MODEL_OVERRIDES) - set(DATASETS):
    raise ValueError("MODEL_OVERRIDES contains unselected datasets")
if any("stitches_per_edge" in values for values in MODEL_OVERRIDES.values()):
    raise ValueError("Set stitch counts only in CONFIGURATIONS")
SCHEDULE = []
for setup in CONFIGURATIONS:
    if set(setup) != {"name", "inducing_points", "patches", "stitches_per_edge"}:
        raise ValueError("Each configuration requires name,inducing_points,patches,stitches_per_edge")
    if not isinstance(setup["name"], str) or not re.fullmatch(r"[A-Za-z0-9_-]+", setup["name"]):
        raise ValueError("Configuration names must be nonempty ASCII letters,digits,hyphens or underscores")
    if any(item["name"] == setup["name"] for item in SCHEDULE):
        raise ValueError("Configuration names must be distinct")
    overrides = {dataset: {**MODEL_OVERRIDES.get(dataset, {}), "stitches_per_edge": setup["stitches_per_edge"]}
                 for dataset in DATASETS}
    settings = call_worker("settings", study="gpw_stitch_rank", device="cuda:0", input_root=INPUT_ROOT,
        datasets=DATASETS, seeds=SEEDS, ranks=RANKS, stitch_modes=STITCH_MODES,
        inducing_points=setup["inducing_points"], patches=setup["patches"],
        epochs=EPOCHS, warmup_epochs=WARMUP_EPOCHS, early_stopping=EARLY_STOPPING,
        model_overrides=overrides, data_overrides=DATA_OVERRIDES,
        progress_every_epochs=PROGRESS_EVERY_EPOCHS, save_inducing_positions=SAVE_INDUCING_POSITIONS)
    SCHEDULE.append({"name": setup["name"], **settings})
if any(item["plan"]["data"] != SCHEDULE[0]["plan"]["data"] for item in SCHEDULE):
    raise ValueError("Capacity plans must have identical dataset configurations")
print("Total runs:", sum(item["total_runs"] for item in SCHEDULE),
      "Maximum total epochs:", sum(item["total_epochs"] for item in SCHEDULE))
_GPW_CELL_RESULT = display(pd.DataFrame([{"configuration": item["name"], "dataset": row["dataset"], **row["config"]}
    for item in SCHEDULE for row in item["plan"]["runs"]]))
_OISST_STAGES['settings'] = True
_GPW_CELL_RESULT


## 3. Check inputs, requested GPU and available disk


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['preflight', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['install', 'settings']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
PREFLIGHT_DIRECTORY = WORK_ROOT / datetime.now(UTC).strftime("size_preflight_%Y%m%dT%H%M%S%fZ")
for item in SCHEDULE:
    print(item["name"], call_worker("preflight", plan=item["plan"], plan_sha256=item["plan_sha256"],
        directory=str(PREFLIGHT_DIRECTORY / item["name"])))
_OISST_STAGES['preflight'] = True


## 4. Prepare once, or explicitly load verified prepared arrays

Keep this stage untouched when changing only the training selection.
To reuse arrays, set PREPARED_ACTION="load" and PREPARED_PATH to the exact
prepared folder, including one from the previous multi-dataset notebook
if configurations/checksums match. Loading never rebuilds failed data.


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['data', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['install', 'settings', 'preflight']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
PREPARED_ACTION = "prepare"
PREPARED_PATH = None
first = SCHEDULE[0]
if PREPARED_ACTION == "prepare":
    PREPARED_DIRECTORY = WORK_ROOT / datetime.now(UTC).strftime("size_prepared_%Y%m%dT%H%M%S%fZ")
    prepared = call_worker("prepare", plan=first["plan"], plan_sha256=first["plan_sha256"],
        directory=str(PREPARED_DIRECTORY), data_root=str(DATA_ROOT))
elif PREPARED_ACTION == "load":
    if PREPARED_PATH is None:
        raise ValueError("Set PREPARED_PATH explicitly for verified data reuse")
    PREPARED_DIRECTORY = Path(PREPARED_PATH)
    prepared = call_worker("load", plan=first["plan"], plan_sha256=first["plan_sha256"],
        directory=str(PREPARED_DIRECTORY))
else:
    raise ValueError("PREPARED_ACTION must be prepare or load")
for item in SCHEDULE[1:]:
    call_worker("load", plan=item["plan"], plan_sha256=item["plan_sha256"], directory=str(PREPARED_DIRECTORY))
_GPW_CELL_RESULT = print(prepared)
_OISST_STAGES['data'] = True
_GPW_CELL_RESULT


## 5. Authorize this batch and train

Set RUN_TRAINING=True after reviewing the complete plan. Training history
updates one display rather than appending a figure every snapshot. Final
plots are saved on disk; completed-run links appear immediately. Failure
or interruption stops the worker. Rerun settings/preflight after changes;
this cell never automatically skips or resumes earlier models.


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['install', 'settings', 'preflight', 'data']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
RUN_TRAINING = False
LIVE_PLOTS = True
if RUN_TRAINING is not True:
    raise RuntimeError("Training disabled. Review all capacity plans,then set RUN_TRAINING=True")
training_view = display("Starting selected capacities...", display_id=True)
metrics_view = display("No completed run yet.", display_id=True)

def show_event(event):
    path = Path(event["directory"])
    if event["kind"] == "snapshot":
        if LIVE_PLOTS:
            training_view.update(Image(filename=str(path / "visuals/training.png")))
    elif event["kind"] == "complete":
        metrics_view.update(pd.DataFrame([{"dataset": event["dataset"], "variant": event["name"], **event["metrics"]}]))
        print(event["dataset"], "/", event["name"], "saved:", event["archive"])
        display(FileLink(event["archive"]))
    else:
        raise ValueError(f"Unknown worker event: {event['kind']}")

STAMP = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
STUDY_DIRECTORY = WORK_ROOT / ("two_sizes_" + STAMP)
STUDY_DIRECTORY.mkdir(parents=True, exist_ok=False)
(STUDY_DIRECTORY / "schedule.json").write_text(json.dumps(SCHEDULE, indent=2, allow_nan=False))
completed = []
for item in SCHEDULE:
    print("Starting configuration:", item["name"])
    batch = STUDY_DIRECTORY / item["name"]
    call_worker("train", plan=item["plan"], plan_sha256=item["plan_sha256"],
        prepared_directory=str(PREPARED_DIRECTORY), directory=str(batch),
        execute_training=RUN_TRAINING, on_event=show_event)
    rows = pd.read_csv(batch / "comparison.csv")
    rows.insert(0, "configuration", item["name"])
    completed.append(rows)
    pd.concat(completed, ignore_index=True).to_csv(STUDY_DIRECTORY / "comparison.csv", index=False)
    archived = call_worker("archive", plan=item["plan"], plan_sha256=item["plan_sha256"],
        directory=str(batch), destination=str(WORK_ROOT / ("gpw_" + item["name"] + "_" + STAMP + ".zip")))
    display(FileLink(archived["archive"]))
_GPW_CELL_RESULT = display(pd.concat(completed, ignore_index=True))
_OISST_STAGES['training'] = True
_GPW_CELL_RESULT


## 6. Archive the completed two-capacity collection


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['archive']:
    _OISST_STAGES[_stage] = False
for _required in ['training']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
import zipfile
ARCHIVE_PATH = WORK_ROOT / ("gpw_two_sizes_" + STAMP + ".zip")
with zipfile.ZipFile(ARCHIVE_PATH, "x", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(p for p in STUDY_DIRECTORY.rglob("*") if p.is_file()):
        bundle.write(path, path.relative_to(STUDY_DIRECTORY))
display(FileLink(str(ARCHIVE_PATH)))
_GPW_CELL_RESULT = print("Save a Kaggle version with outputs or download archives before session storage is lost.")
_OISST_STAGES['archive'] = True
_GPW_CELL_RESULT
